In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import zipfile, os

zip_path = "/content/drive/MyDrive/agro-spray/betel.zip"
extract_path = "/content/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Done")

Done


In [3]:
for split in ['train', 'val', 'test']:
    for cls in ['healthy', 'diseased']:
        path = f"/content/data/betel/{split}/{cls}"
        count = len(os.listdir(path)) if os.path.exists(path) else "PATH NOT FOUND"
        print(f"{split}/{cls}: {count}")

train/healthy: 757
train/diseased: 669
val/healthy: 162
val/diseased: 143
test/healthy: 162
test/diseased: 143


In [4]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
else:
    print("Device: CPU")

GPU available: True
Device: Tesla T4


In [5]:
import os
print(os.listdir("/content/data/betel"))

['val', 'train', 'test']


In [6]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [8]:
train_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])
val_transforms = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

In [9]:
train_dataset = datasets.ImageFolder("/content/data/betel/train",transform = train_transforms)
val_dataset = datasets.ImageFolder("/content/data/betel/val",transform = val_transforms)

In [10]:
print(train_dataset.classes)

['diseased', 'healthy']


In [11]:
len(train_dataset),len(val_dataset)

(1426, 305)

In [12]:
train_loader = DataLoader(train_dataset,batch_size = 32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle=False)

In [13]:
model = models.resnet18(weights= models.ResNet18_Weights.IMAGENET1K_V1)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 169MB/s]


In [14]:
#model

In [15]:
for param in model.parameters():
    param.requires_grad = False

In [16]:
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

In [19]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr = 0.001)

In [21]:
NUM_EPOCHS = 10

for epoch in range(NUM_EPOCHS):

    # Training phase
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | "
          f"Train loss: {train_loss/train_total:.4f} "
          f"acc: {train_correct/train_total:.3f} | "
          f"Val loss: {val_loss/val_total:.4f} "
          f"acc: {val_correct/val_total:.3f}")

Epoch 1/10 | Train loss: 0.2128 acc: 0.911 | Val loss: 0.2726 acc: 0.892
Epoch 2/10 | Train loss: 0.1951 acc: 0.928 | Val loss: 0.2655 acc: 0.895
Epoch 3/10 | Train loss: 0.2104 acc: 0.919 | Val loss: 0.2635 acc: 0.895
Epoch 4/10 | Train loss: 0.2039 acc: 0.920 | Val loss: 0.2626 acc: 0.882
Epoch 5/10 | Train loss: 0.1919 acc: 0.931 | Val loss: 0.2950 acc: 0.875
Epoch 6/10 | Train loss: 0.2006 acc: 0.928 | Val loss: 0.2616 acc: 0.882
Epoch 7/10 | Train loss: 0.1890 acc: 0.928 | Val loss: 0.2530 acc: 0.889
Epoch 8/10 | Train loss: 0.1971 acc: 0.926 | Val loss: 0.2523 acc: 0.889
Epoch 9/10 | Train loss: 0.1934 acc: 0.926 | Val loss: 0.2603 acc: 0.875
Epoch 10/10 | Train loss: 0.1854 acc: 0.926 | Val loss: 0.2473 acc: 0.898


In [23]:
torch.save({
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'val_acc': 0.898,
}, "/content/drive/MyDrive/agro-spray/betel_checkpoint.pth")

In [24]:
checkpoint = torch.load("/content/drive/MyDrive/agro-spray/betel_checkpoint.pth")
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoch_10 = checkpoint['epoch']
val_acc_10 = checkpoint['val_acc']
print(epoch_10, val_acc_10)

10 0.898


In [25]:
NUM_EPOCHS = 20
best_acc = val_acc_10
best_epoch = epoch_10
train_loss_list = []
train_acc_list = []
val_loss_list = []
val_acc_list = []
patience = 5
counter = 0
for epoch in range(NUM_EPOCHS):

    # Training phase
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_function(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    # Validation phase
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = loss_function(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_correct += (predicted == labels).sum().item()
            val_total += labels.size(0)
    val_acc = val_correct / val_total
    train_loss_list.append(train_loss/train_total)
    train_acc_list.append(train_correct/train_total)
    val_loss_list.append(val_loss/val_total)
    val_acc_list.append(val_correct/val_total)

    print(f"Epoch {epoch+11}/{NUM_EPOCHS+10} | "
          f"Train loss: {train_loss/train_total:.4f} "
          f"acc: {train_correct/train_total:.3f} | "
          f"Val loss: {val_loss/val_total:.4f} "
          f"acc: {val_correct/val_total:.3f}")
    if val_acc > best_acc:
        torch.save({
            'epoch': (epoch+11),
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc}, "/content/drive/MyDrive/agro-spray/betel_checkpoint.pth")
        best_acc = val_acc
        best_epoch = epoch+11
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping! Best model saved at {best_epoch}")
            break


df = pd.DataFrame({"epoch": list(range(11,11+len(train_loss_list))),'train_loss': train_loss_list, 'train_acc': train_acc_list, 'val_loss': val_loss_list, 'val_acc': val_acc_list})
df.to_csv("/content/drive/MyDrive/agro-spray/betel_history_11_30.csv",index=False)

Epoch 11/30 | Train loss: 0.1702 acc: 0.937 | Val loss: 0.2485 acc: 0.908
Epoch 12/30 | Train loss: 0.1940 acc: 0.927 | Val loss: 0.2509 acc: 0.905
Epoch 13/30 | Train loss: 0.1976 acc: 0.921 | Val loss: 0.2577 acc: 0.898
Epoch 14/30 | Train loss: 0.1988 acc: 0.917 | Val loss: 0.2861 acc: 0.892
Epoch 15/30 | Train loss: 0.2145 acc: 0.910 | Val loss: 0.3209 acc: 0.879
Epoch 16/30 | Train loss: 0.1777 acc: 0.931 | Val loss: 0.2544 acc: 0.885
Early stopping! Best model saved at 11
